# 超参数搜索：从第一次调参到自定义目标

本教程使用 `examples/hscredit_yyp.xlsx` 的真实放款数据，按顺序展示：最小调用、搜索空间、普通指标函数、可复用 `BaseMetric`、内置和自定义 loss、金额权重、多目标、时间验证、逐折复盘、保存续跑。

依赖：`pip install -e ".[boost,tune,dev]"`；从仓库根目录或 `examples` 目录启动，按顺序执行。每个例子只运行少量 trial，便于理解接口；正式建模应根据数据、算力和业务稳定性要求调整。

三个参数各司其职：`search_space` 决定尝试哪些模型参数，`loss` 决定训练怎样计算梯度，`metric` 决定哪个候选模型更好。`n_trials=3, cv=3` 通常要做 9 次折训练，取最佳模型时另做一次最终重训。**测试集不参与参数选择或早停。**

完整说明见 [调参指南](../docs/articles/tuning-guide.md)；各损失业务含义见 [损失指南](../docs/articles/losses-guide.md) 和 [losses_workflow.ipynb](losses_workflow.ipynb)。

In [1]:
from pathlib import Path
from tempfile import mkdtemp
import sys

repository_root = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "hscredit").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("请从仓库根目录或 examples 目录启动 Notebook")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import numpy as np
import pandas as pd
import optuna
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from hscredit import LightGBM, ModelTuner
from hscredit.core.models.tuning import Integer, Real, Categorical
from hscredit.core.models.losses import (
    BaseLoss, BaseMetric, FocalLoss, WeightedBCELoss, AUCMetric, make_metric,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
FEATURES = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
data = pd.read_excel(repository_root / "examples" / "hscredit_yyp.xlsx")
data = data.loc[data["FPD"].isin([0, 1])].copy()
X = data[FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y = data["FPD"].astype(int)
amount = pd.to_numeric(data["放款金额"], errors="raise")
X_train, X_test, y_train, y_test, amount_train, amount_test = train_test_split(
    X, y, amount, test_size=0.3, stratify=y, random_state=42,
)
print(f"训练样本数：{len(X_train)}；测试样本数：{len(X_test)}")

训练样本数：679；测试样本数：291


## 1. 已有模型：用 `.tune()` 获取最佳模型

最少只需模型、搜索空间和评估指标。`metric="auc"` 自动最大化，`metric="log_loss"` 自动最小化，不必重复写方向。已有模型的其他设置继续保留；搜索参数覆盖原配置，显式 `fixed_params` 的优先级最高。

`tune()` 返回已经重训好的最佳模型。原模型与返回模型均通过 `.tuner` 查看同一次搜索，原对象本身不会被替换为最佳模型。

In [2]:
model = LightGBM(n_estimators=30, num_leaves=7, n_jobs=1, random_state=42)
best = model.tune(
    X_train, y_train,
    search_space={"max_depth": [3, 4], "learning_rate": [0.03, 0.08]},
    metric="auc", cv=3, n_trials=3, n_jobs=1,
    retention="predictions", show_progress_bar=False,
)
tuner = best.tuner
print("最佳参数：", tuner.best_params_)
print(f"CV 平均 AUC：{tuner.best_score_:.6f}")
print(f"独立测试 AUC：{AUCMetric().evaluate(y_test, best.predict_proba(X_test)):.6f}")
assert tuner.get_best_model() is best

最佳参数： {'max_depth': 4, 'learning_rate': 0.03, 'boosting_type': 'gbdt', 'colsample_bytree': 1.0, 'early_stopping_metric': None, 'early_stopping_rounds': None, 'eval_metric': None, 'first_metric_only': True, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0, 'n_estimators': 30, 'n_jobs': 1, 'num_leaves': 7, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0, 'reg_lambda': 0, 'scale_pos_weight': 1, 'scorecard_params': None, 'subsample': 1.0, 'validation_fraction': 0.2, 'verbose': False, 'target': None, 'history_policy': 'summary', 'max_history': 20, 'weight_type': 'cost'}
CV 平均 AUC：0.570481
独立测试 AUC：0.546244


## 2. 独立调参器：理解 `fit` 的返回值

`ModelTuner.fit()` 为兼容旧 API 返回**最佳参数字典**，不是调参器或模型。先保存 `tuner`，再调用 `get_best_model()`。首次取模型会在全部输入训练数据上重训并缓存，后续调用复用缓存；`refit=True` 显式要求再训练。

以下辅助函数仅减少本 Notebook 的重复配置。每次都新建调参器；不同数据、指标或 CV 口径使用不同 Study。

In [3]:
def make_tuner(metric="auc", **kwargs):
    """创建小规模演示搜索器，默认不做内部早停。"""
    options = dict(
        model_class=LightGBM,
        search_space={"max_depth": [3, 4]},
        fixed_params={"n_estimators": 25, "num_leaves": 7, "validation_fraction": 0.0},
        metric=metric, cv=3, n_jobs=1, random_state=42,
        early_stopping_rounds=None, retention="predictions",
    )
    options.update(kwargs)
    return ModelTuner(**options)

loss_score_tuner = make_tuner(metric="log_loss")
best_params = loss_score_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
assert isinstance(best_params, dict)
logloss_best = loss_score_tuner.get_best_model()
print(f"CV 平均交叉熵：{loss_score_tuner.best_score_:.6f}；越小越好")

CV 平均交叉熵：0.404221；越小越好


## 3. 搜索空间：先学三种声明，再学条件空间

`Integer` 是整数范围，`Real` 是连续范围，`Categorical` 是离散选项。列表也是离散候选，**列表声明不意味着穷举所有组合**；默认仍由 TPE 采样。固定参数放入 `fixed_params`，避免当作搜索维度。

`Integer(3, 4)` 包含两端；学习率跨数量级时可用 `Real(..., prior="log-uniform")`。调参器的 `n_jobs` 主要控制模型并行，trial 按顺序运行。

In [4]:
space = {
    "max_depth": Integer(3, 4),
    "learning_rate": Real(0.02, 0.1, prior="log-uniform"),
    "reg_lambda": Categorical([0.0, 1.0, 3.0]),
}
space_tuner = make_tuner(search_space=space)
space_tuner.enqueue_trial({"max_depth": 3, "learning_rate": 0.05, "reg_lambda": 1.0})
space_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
print("搜索空间示例最佳参数：", space_tuner.best_params_)

搜索空间示例最佳参数： {'max_depth': 3, 'learning_rate': 0.09237421878009823, 'reg_lambda': 0.0, 'n_estimators': 25, 'num_leaves': 7, 'validation_fraction': 0.0}


复杂关联用 `search_space(trial) -> dict`：例如叶子数不能超过 `2 ** max_depth`，或某模型参数仅在另一参数取特定值时出现。返回的键必须是模型构造参数；Trial 内部参数名也会保留在 Study 中。

In [5]:
def conditional_space(trial):
    depth = trial.suggest_int("max_depth", 2, 4)
    return {
        "max_depth": depth,
        "num_leaves": trial.suggest_int("num_leaves", 3, min(15, 2 ** depth)),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
    }

conditional_tuner = make_tuner(
    search_space=conditional_space,
    fixed_params={"n_estimators": 25, "validation_fraction": 0.0},
)
conditional_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
print("条件空间最佳参数：", conditional_tuner.best_params_)

条件空间最佳参数： {'max_depth': 3, 'num_leaves': 3, 'learning_rate': 0.025707833957860277, 'n_estimators': 25, 'validation_fraction': 0.0}


## 4. 自定义评估指标：普通函数只需加名称和方向

评估函数签名是 `(y_true, y_prob, sample_weight=None, **固定参数) -> float`；收到的 `y_prob` 是一维坏样本概率，不是 logits，也不是两列概率。

用 `make_metric` 包装后，离线评估、训练监控、ModelTuner 和 sklearn scorer 都可复用同一个对象。没有额外参数时可以直接使用 sklearn 的 `brier_score_loss` 等概率指标。

In [6]:
def probability_error(y_true, y_prob, sample_weight=None, scale=1.0):
    """均方概率误差，scale 是固定业务系数。"""
    error = (np.asarray(y_prob) - np.asarray(y_true)) ** 2
    return float(scale * np.average(error, weights=sample_weight))

brier = make_metric(
    probability_error, name="均方概率误差", greater_is_better=False, scale=1.0,
)
function_tuner = make_tuner(metric=brier)
function_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
function_model = function_tuner.get_best_model()
print(f"自定义指标 CV：{function_tuner.best_score_:.6f}")
print(f"自定义指标测试值：{brier.evaluate(y_test, function_model.predict_proba(X_test)):.6f}")
assert brier.direction == "minimize"

# 也可直接传裸函数，但必须显式声明方向，避免误把误差最大化。
raw_function_tuner = make_tuner(metric=probability_error, direction="minimize")
raw_function_tuner.fit(X_train, y_train, n_trials=1, show_progress_bar=False)

自定义指标 CV：0.119921
自定义指标测试值：0.121233


## 5. 需要封装状态：继承 BaseMetric

指标需要多个参数、方法或业务语义时再定义类。实现 `__call__` 即可，基类 `evaluate()` 会校验输入、支持两列概率及显式原始分数转换。额外权重要在 `__call__` 中明确接收并使用，不能静默忽略。

In [7]:
class ProbabilityError(BaseMetric):
    """可复用的均方概率误差。"""
    def __init__(self):
        super().__init__(name="均方概率误差", greater_is_better=False)

    def __call__(self, y_true, y_pred, sample_weight=None):
        error = (np.asarray(y_pred) - np.asarray(y_true)) ** 2
        return float(np.average(error, weights=sample_weight))

custom_metric = ProbabilityError()
class_tuner = make_tuner(metric=custom_metric)
class_tuner.fit(X_train, y_train, n_trials=1, show_progress_bar=False)
assert np.isclose(custom_metric.evaluate(y_test, best.predict_proba(X_test)), brier.evaluate(y_test, best.predict_proba(X_test)))
print("BaseMetric 自动采用方向：", class_tuner.directions)

BaseMetric 自动采用方向： ['minimize']


## 6. 自定义 loss 训练，固定 metric 选择模型

`loss=...` 便捷参数支持 HSCredit `LightGBM` / `XGBoost` / `CatBoost` 包装器。设 `metric=None` 会使用 `loss.metric()`；设 `metric="auc"` 则训练优化该 loss、搜索最大化 AUC。

`ModelTuner(objective=...)` 是历史上的**搜索评分目标**别名，不是模型训练 loss。原生 sklearn estimator 请在模型构造器或 `fixed_params` 中按该框架签名配置训练目标。

In [8]:
loss = FocalLoss(alpha=0.75, gamma=2.0)
focal_tuner = make_tuner(loss=loss, metric=None)
focal_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
focal_model = focal_tuner.get_best_model()
print(f"Focal CV 损失：{focal_tuner.best_score_:.6f}")
print(f"Focal 测试损失：{loss.evaluate(y_test, focal_model.predict_proba(X_test)):.6f}")
assert focal_tuner.directions == ["minimize"]

Focal CV 损失：0.051486
Focal 测试损失：0.052866


搜索 loss 自身参数时，在条件搜索函数返回模型的 `objective` 对象。**评估指标必须固定**：下例搜索 alpha/gamma，但始终用 AUC 比较。不能让每个 trial 的 loss 参数同时改变评价标尺。

In [9]:
def loss_parameter_space(trial):
    return {
        "objective": FocalLoss(
            alpha=trial.suggest_float("alpha", 0.4, 0.8),
            gamma=trial.suggest_float("gamma", 0.5, 2.0),
        ),
        "max_depth": trial.suggest_int("max_depth", 3, 4),
    }

loss_parameter_tuner = make_tuner(metric=AUCMetric(), search_space=loss_parameter_space)
loss_parameter_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
print(f"搜索 loss 参数后的 CV AUC：{loss_parameter_tuner.best_score_:.6f}")

搜索 loss 参数后的 CV AUC：0.591070


## 7. 从零实现一个自定义训练 loss

Brier loss 对概率求导，适配器负责转换为原始分数导数。逐样本贡献的均值与 `__call__` 相同，`gradient` / `hessian` 是“样本数 × 平均损失”的导数；`is_additive=True` 表示允许分批与外部权重。

In [10]:
class BrierLoss(BaseLoss):
    """均方概率误差训练目标。"""
    is_additive = True

    def __init__(self):
        super().__init__("均方概率损失")

    def loss_values(self, y_true, y_pred):
        return (np.asarray(y_pred) - np.asarray(y_true)) ** 2

    def __call__(self, y_true, y_pred):
        return float(np.mean(self.loss_values(y_true, y_pred)))

    def gradient(self, y_true, y_pred):
        return 2 * (np.asarray(y_pred) - np.asarray(y_true))

    def hessian(self, y_true, y_pred):
        return np.full(len(y_true), 2.0)

custom_loss_tuner = make_tuner(loss=BrierLoss(), metric=brier)
custom_loss_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
custom_loss_model = custom_loss_tuner.get_best_model()
print(f"自定义 loss 模型测试 Brier：{brier.evaluate(y_test, custom_loss_model.predict_proba(X_test)):.6f}")

自定义 loss 模型测试 Brier：0.133298


## 8. 金额加权：区分训练权重和评估权重

`sample_weight` 控制训练；`evaluation_weight` 控制 CV 选优。只传训练权重不会暗中改变评估口径。两者都应与 `X_train` 按行位置对齐，调参器按训练/验证折分别切分。

需要金额加权 BCE 时，使用普通 `WeightedBCELoss` 加权即可。不把全量金额数组绑定进跨折复用的 `AmountWeightedLoss`；绑定数组需要手动逐折创建损失和指标。

In [11]:
weighted_loss = WeightedBCELoss()
weighted_tuner = make_tuner(loss=weighted_loss, metric=weighted_loss.metric())
weight_train = amount_train.to_numpy(dtype=float)
weighted_tuner.fit(
    X_train, y_train, sample_weight=weight_train, evaluation_weight=weight_train,
    n_trials=2, show_progress_bar=False,
)
weighted_model = weighted_tuner.get_best_model()
weighted_test = weighted_loss.evaluate(
    y_test, weighted_model.predict_proba(X_test), sample_weight=amount_test.to_numpy(dtype=float),
)
print(f"CV 金额加权 BCE：{weighted_tuner.best_score_:.6f}；测试值：{weighted_test:.6f}")

CV 金额加权 BCE：0.402007；测试值：0.433242


## 9. sklearn Pipeline：预处理在每折内部拟合

模型可传 estimator 实例，Pipeline 内的参数用 `步骤名__参数`。使用自定义空间，不依赖 HSCredit 特定模型的默认空间。不要先对全量数据拟合填充器/标准化器再做 CV。

In [12]:
pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))
pipeline_tuner = ModelTuner(
    pipeline, search_space={"logisticregression__C": Real(0.01, 10.0, prior="log-uniform")},
    metric=brier, cv=3, n_jobs=1, random_state=42, retention="predictions",
)
pipeline_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
pipeline_best = pipeline_tuner.get_best_model()
print(f"Pipeline 测试 Brier：{brier.evaluate(y_test, pipeline_best.predict_proba(X_test)):.6f}")

Pipeline 测试 Brier：0.118045


## 10. 多目标：保留所有不能同时被超越的候选

`metric=[...]` 声明多个目标，各指标自动给出方向。这里同时最大化 AUC、最小化 Brier。`get_pareto_front()` 返回 Pareto 候选；默认 `get_best_model()` 按指标列表顺序优先第一个指标，同分时看下一个，不能代替业务权衡。

In [13]:
multi_tuner = make_tuner(metric=[AUCMetric(), brier], sampler="nsgaii")
multi_tuner.fit(X_train, y_train, n_trials=3, show_progress_bar=False)
pareto = pd.DataFrame([
    {"试验编号": trial.number, "AUC": trial.values[0], "均方概率误差": trial.values[1]}
    for trial in multi_tuner.get_pareto_front()
])
print(pareto.to_string(index=False))
print("默认最优试验：", multi_tuner.best_trial_.number)
assert multi_tuner.directions == ["maximize", "minimize"]

 试验编号    AUC  均方概率误差
    0 0.5891  0.1209
    1 0.5835  0.1199
    2 0.5835  0.1199
默认最优试验： 0


## 11. 时间交叉验证：按放款时间模拟向未来预测

按 `放款时间` 排序，最后 20% 保留作时间外测试。`TimeSeriesSplit` 只用过去训练、未来验证，不打乱时间。AUC 要求每个验证折同时含好坏两类，先验证这个条件。时间 CV 的最早一段不会获得 OOF 预测，输出保留缺失值是正常现象。

客户/门店分组场景则使用 `cv=GroupKFold(...)` 并在 `fit(..., groups=分组数组)` 中传入真实组标识。

In [14]:
time_data = data.assign(放款时间=pd.to_datetime(data["放款时间"], errors="coerce"))
time_data = time_data.dropna(subset=["放款时间"]).sort_values("放款时间", kind="stable")
cut = int(len(time_data) * 0.8)
time_train, time_test = time_data.iloc[:cut], time_data.iloc[cut:]
X_time = time_train[FEATURES].apply(pd.to_numeric, errors="coerce")
y_time = time_train["FPD"].astype(int)
time_cv = TimeSeriesSplit(n_splits=3)
assert all(y_time.iloc[valid].nunique() == 2 for _, valid in time_cv.split(X_time))
time_tuner = make_tuner(cv=time_cv)
time_tuner.fit(X_time, y_time, n_trials=2, show_progress_bar=False)
time_model = time_tuner.get_best_model()
time_probability = time_model.predict_proba(time_test[FEATURES].apply(pd.to_numeric, errors="coerce"))
print(f"时间外测试 AUC：{AUCMetric().evaluate(time_test['FPD'], time_probability):.6f}")

时间外测试 AUC：0.438185


## 12. 结果复盘、OOF 和复训

`best_score_` 是外层 CV 各折指标均值，不是最终模型在训练集或测试集的得分。`get_trial_result()` 查看逐折记录，`get_oof_predictions()` 查看折外预测；OOF 只用于训练样本的折外估计。

`get_optimization_history()` 保留兼容的 Optuna 历史字段；`retention="predictions"` 保存预测但不保留各折模型，适合本例。最终模型另有缓存。

In [15]:
record = tuner.get_trial_result(tuner.best_trial_.number)
print("最佳试验状态：", record["状态"], "；折数：", len(record["各折"]))
oof = tuner.get_oof_predictions()
print("OOF 列：", list(oof.columns), "；行数：", len(oof))
assert len(oof) == len(X_train)
print(tuner.get_optimization_history().head().to_string(index=False))
assert tuner.get_best_model() is best
refitted = tuner.get_best_model(refit=True)
print(f"显式复训后的测试 AUC：{AUCMetric().evaluate(y_test, refitted.predict_proba(X_test)):.6f}")

最佳试验状态： 完成 ；折数： 3
OOF 列： ['样本位置', '真实标签', '预测概率', '验证次数'] ；行数： 679
 number  value_AUC             datetime_start          datetime_complete               duration  params_learning_rate  params_max_depth                                                                         user_attrs_各折指标    state  LIFT@1%  LIFT@3%  LIFT@5%  LIFT@10%
      0     0.5705 2026-10-09 00:04:51.275004 2026-10-09 00:04:51.309793 0 days 00:00:00.034789                0.0300                 4 [{'AUC': 0.5815705128205129}, {'AUC': 0.5485663659793814}, {'AUC': 0.5813068651778329}] COMPLETE   1.9353   2.8887   1.9834    1.7251
      1     0.5650 2026-10-09 00:04:51.309793 2026-10-09 00:04:51.331527 0 days 00:00:00.021734                0.0800                 3 [{'AUC': 0.5920673076923078}, {'AUC': 0.5367268041237113}, {'AUC': 0.5660876757650951}] COMPLETE   1.6201   2.2710   1.8844    1.6541
      2     0.5638 2026-10-09 00:04:51.331527 2026-10-09 00:04:51.353036 0 days 00:00:00.021509                0.0800      

## 13. 保存与恢复：完整对象、SQLite 和推理模型

完整 `tuner.save()` 包含搜索配置、原始训练数据、指标函数、Study 和当前保留的结果，适合继续同一个任务。SQLite 保存 Optuna 的 trial 状态，不自动保存 Python 模型和预测；需要跨进程读取折结果时同时设置 `artifact_dir`。

此例在新临时目录写文件，不覆盖现有结果。恢复后传入相同数据和 CV/指标配置再 `fit(n_trials=1)` 表示**追加 1 次 trial**。数据、行顺序或评估口径发生变化，应新建调参器与 Study。

In [16]:
output_dir = Path(mkdtemp(prefix="hscredit_tuning_workflow_"))
storage_url = "sqlite:///" + (output_dir / "search.db").as_posix()
persistent_tuner = make_tuner(
    metric=brier, storage=storage_url, study_name="FPD概率误差演示",
    artifact_dir=str(output_dir / "trials"), retention="disk",
)
persistent_tuner.fit(X_train, y_train, n_trials=2, show_progress_bar=False)
persistent_best = persistent_tuner.get_best_model()
tuner_path = output_dir / "tuner.joblib"
persistent_tuner.save(tuner_path)
restored = ModelTuner.load(tuner_path)
np.testing.assert_allclose(restored.get_best_model().predict_proba(X_test), persistent_best.predict_proba(X_test))
old_count = len(restored.study_.trials)
restored.fit(X_train, y_train, n_trials=1, show_progress_bar=False)
assert len(restored.study_.trials) == old_count + 1
restored.get_best_model().save_inference(output_dir / "最佳预测模型.joblib")
print("验证输出目录：", output_dir)
print("恢复后试验总数：", len(restored.study_.trials))

恢复后试验总数： 3


## 14. 可视化与扩展入口

完整搜索过程保留在 `tuner.get_study()`。下列方法生成图对象；在 Jupyter 末行展示或调用 `.show()` 即可。少量 trial 的参数重要性不稳定，本例只构造历史图并核验对象。

更复杂的训练逻辑可使用 `fit_params_factory(trial, fold_index)` 每折创建训练回调，或 `trial_objective(trial)` 完全接管试验。后者需要自己管理切分、训练、评价与记录，不享有自动逐折保存。

In [17]:
figure = tuner.visualization.plot_optimization_history()
assert len(figure.data) > 0
print("已生成搜索历史图，可运行 figure.show() 查看。")

已生成搜索历史图，可运行 figure.show() 查看。


## 15. 常见误用检查

- 调参结果“反着选”：裸函数必须声明方向；指标对象通过 `greater_is_better` 声明一次。
- `metric="log_loss"` 是正的交叉熵，越小越好；旧 `"logloss"` / `"neg_log_loss"` 是负值，越大越好，保留作兼容。
- `loss` 决定训练，`metric` 决定候选优劣，模型的 `eval_metric` 决定训练监控/早停；三者可以不同。
- `ModelTuner(objective=...)` 是旧评分入口；自定义训练用 `loss=...`，完全接管 trial 才用 `trial_objective=...`。
- 自定义空间的名字必须是模型参数名；Pipeline 用 `步骤名__参数`；列表空间不是自动穷举。
- 不同 loss 参数的损失数值不可直接比大小；调 loss 参数时使用固定 AUC、固定 Brier 或固定业务指标。
- 加权训练和加权评价分别传 `sample_weight`、`evaluation_weight`；自定义指标必须支持权重。
- 测试集不传入搜索/早停；绑定金额等行级数组不能直接跨折复用。
- 只保存 SQLite 不能恢复 Python 模型；完整制品、折制品目录、外部数据库各自保留。